# 中证800 V64：Walk-Forward OOS 鲁棒性验证

目的：验证当前主线算法是否在不同训练窗口和后续 OOS 阶段保持稳定，而不是只在某个固定 cutoff 上有效。

固定不变：
- full 特征集
- V46/V61 LightGBM 参数，固定 120 轮
- legacy_unsealed 训练边界
- `top8_board_cap` 组合构建：持仓 8 只，创业板最多 3 只，科创板最多 2 只

本 notebook 不做新模型发明，也不做大规模调参。它输出的是：
1. walk-forward fold 明细
2. 月度 OOS 组合表现
3. fold/年度稳定性汇总
4. 简洁 pass/fail 鲁棒性判断

注意：V64 是离线 OOS/proxy 验证；最终仍需要用 V61 的 JQ-like 日频账户路径验收胜出的模型和规则。


In [ ]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        for i, item in enumerate(iterable, 1):
            if i == 1 or (total is not None and i == total) or i % max(1, int((total or 100) / 20)) == 0:
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()

# =========================
# Config
# =========================
DATA_PATH = "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"
OUT_DIR = Path("csi800_ml_v64_walk_forward_oos_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"
BOUNDARY_POLICY = "legacy_unsealed_q4"
USE_LEGACY_UNSEALED_BOUNDARY = True

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
STOCK_NUM = 8
PORTFOLIO_RULE = "top8_board_cap"
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])
RANDOM_SIM_N = 500
RANDOM_SEED = 42

# Walk-forward 设置：6 个月 OOS，半年滚动一次。anchored 模拟持续扩训；rolling60 模拟只保留近 5 年样本。
MIN_TRAIN_MONTHS = 36
TEST_MONTHS = 6
STEP_MONTHS = 6
MIN_TEST_MONTHS = 3
FOLD_SETUPS = [
    {"fold_set": "anchored_36m_train_6m_oos", "rolling_train_months": None},
    {"fold_set": "rolling60m_train_6m_oos", "rolling_train_months": 60},
]

# 简洁鲁棒性门槛，不作为调参目标，只作为红黄绿判断。
PASS_MIN_MONTHS = 18
PASS_MIN_WIN_RATE = 0.55
PASS_MIN_POSITIVE_FOLD_RATE = 0.60
PASS_MIN_RANDOM_PERCENTILE = 0.55
PASS_MAX_DRAWDOWN = -0.15

print("DATA_PATH:", DATA_PATH)
print("OUT_DIR:", OUT_DIR.resolve())
print("portfolio:", PORTFOLIO_RULE, "stock_num", STOCK_NUM, "board_caps", BOARD_CAPS_TEXT)


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 1.0,
    "bagging_freq": 0,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "verbosity": -1,
    "num_threads": 4,
}


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return (1.0 + s).cumprod() if len(s) else pd.Series(dtype=float)


def calc_drawdown_from_returns(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_returns(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "ann_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "sharpe12": np.nan}
    std = s.std(ddof=1)
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "ann_ret": float((1.0 + s).prod() ** (12.0 / len(s)) - 1.0) if len(s) else np.nan,
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_drawdown_from_returns(s),
        "sharpe12": float(s.mean() / std * np.sqrt(12)) if len(s) > 1 and std > 0 else np.nan,
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    usable = []
    for col in unique_keep_order(candidate_cols):
        if col not in train_df.columns:
            continue
        s = pd.to_numeric(train_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if s.notnull().sum() < max(20, int(len(train_df) * 0.05)):
            continue
        if s.nunique(dropna=True) < 2:
            continue
        usable.append(col)
    if len(usable) == 0:
        raise ValueError("no usable features")
    comps = build_corr_components(train_df, usable, CORR_THRESHOLD)
    ic_map = {}
    for col in usable:
        ic_map[col] = abs(safe_rank_ic(train_df[col], train_df[TARGET_COL]))
    kept = []
    removed = []
    for comp in comps:
        if len(comp) == 1:
            kept.append(comp[0])
            continue
        comp_sorted = sorted(comp, key=lambda x: (ic_map.get(x, 0.0), -usable.index(x)), reverse=True)
        kept.append(comp_sorted[0])
        removed.extend(comp_sorted[1:])
    return unique_keep_order(kept), removed


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    if fill_values is None:
        fill_values = X.median(numeric_only=True).to_dict()
    X = X.fillna(fill_values).fillna(0)
    y = pd.to_numeric(d[target_col], errors="coerce").astype(float)
    ok = y.notnull().values
    return X.loc[ok, feature_cols], y.loc[ok], fill_values, d.loc[ok].index


def split_diag_valid(train_df, valid_months=6):
    months = sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= valid_months + 6:
        return train_df.copy(), train_df.copy()
    valid_set = set(months[-valid_months:])
    fit = train_df[~train_df[DATE_COL].isin(valid_set)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_set)].copy()
    return fit, valid


In [ ]:
def first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def load_dataset(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and STOCK_COL not in df.columns:
        df = df.rename(columns={"code": STOCK_COL})
    for col in [DATE_COL, "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()
    if STOCK_COL not in df.columns:
        raise ValueError("missing stock column")
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("missing target: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


def infer_return_columns(df):
    raw_col = first_existing(df.columns, ["raw_return_1m", "stock_return_1m", "return_1m", "next_return_1m"])
    bench_col = first_existing(df.columns, ["benchmark_csi800_1m", "benchmark_000906_1m", "benchmark_alla_1m", "cum_csi800_1m"])
    alpha_col = TARGET_COL if TARGET_COL in df.columns else None
    if raw_col is None and alpha_col is not None and bench_col is not None:
        df["_v64_raw_return_1m"] = df[alpha_col] + df[bench_col]
        raw_col = "_v64_raw_return_1m"
    return raw_col, bench_col, alpha_col


def get_month_benchmark_return(month_df, bench_col):
    if bench_col is None:
        return np.nan
    s = pd.to_numeric(month_df[bench_col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.iloc[0]) if len(s) else np.nan


df_all = load_dataset(DATA_PATH)
RAW_RET_COL, BENCH_RET_COL, ALPHA_RET_COL = infer_return_columns(df_all)
available_features = [c for c in FULL_FEATURE_COLS if c in df_all.columns]
missing_features = [c for c in FULL_FEATURE_COLS if c not in df_all.columns]
months_all = sorted(pd.to_datetime(df_all[DATE_COL].dropna().unique()))

print("loaded:", df_all.shape)
print("date range:", pd.Timestamp(months_all[0]).date(), "->", pd.Timestamp(months_all[-1]).date(), "months", len(months_all))
print("raw:", RAW_RET_COL, "bench:", BENCH_RET_COL, "alpha:", ALPHA_RET_COL)
print("features available/missing:", len(available_features), len(missing_features))
if missing_features:
    print("missing features:", ",".join(missing_features))
display(df_all[[TARGET_COL]].describe())


In [ ]:
def make_forward_folds(months, fold_set, min_train_months, test_months, step_months, rolling_train_months=None):
    months = list(sorted(pd.to_datetime(months)))
    rows = []
    start_idx = int(min_train_months)
    fold_no = 0
    while start_idx < len(months):
        test_slice = months[start_idx:start_idx + int(test_months)]
        if len(test_slice) < MIN_TEST_MONTHS:
            break
        train_end_idx = start_idx
        if rolling_train_months is None:
            train_slice = months[:train_end_idx]
        else:
            train_slice = months[max(0, train_end_idx - int(rolling_train_months)):train_end_idx]
        if len(train_slice) < min_train_months:
            start_idx += int(step_months)
            continue
        fold_no += 1
        rows.append({
            "fold_set": fold_set,
            "fold_id": "%s_%02d" % (fold_set, fold_no),
            "train_start": train_slice[0],
            "train_end": train_slice[-1],
            "test_start": test_slice[0],
            "test_end": test_slice[-1],
            "train_months": len(train_slice),
            "test_months": len(test_slice),
            "rolling_train_months": rolling_train_months if rolling_train_months is not None else np.nan,
        })
        start_idx += int(step_months)
    return rows


fold_rows = []
for setup in FOLD_SETUPS:
    fold_rows.extend(make_forward_folds(
        months_all,
        setup["fold_set"],
        MIN_TRAIN_MONTHS,
        TEST_MONTHS,
        STEP_MONTHS,
        setup.get("rolling_train_months"),
    ))
fold_plan_df = pd.DataFrame(fold_rows)
if fold_plan_df.empty:
    raise ValueError("empty fold plan; check date range or MIN_TRAIN_MONTHS")

display(fold_plan_df)
fold_plan_df.to_csv(OUT_DIR / "v64_fold_plan.csv", index=False)


In [ ]:
def board_type(stock):
    s = str(stock)
    if s.startswith("30"):
        return "chinext"
    if s.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    if not board_caps:
        return True
    b = board_type(stock)
    if b not in board_caps:
        return True
    return sum(1 for x in selected if board_type(x) == b) < int(board_caps[b])


def build_board_capped_targets(sorted_stocks, target_num=STOCK_NUM, board_caps=BOARD_CAPS):
    selected = []
    for stock in sorted_stocks:
        if stock in selected:
            continue
        if board_cap_allows(selected, stock, board_caps):
            selected.append(stock)
        if len(selected) >= target_num:
            return selected[:target_num]
    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
        if len(selected) >= target_num:
            break
    return selected[:target_num]


def summarize_target_board(targets):
    counts = {"board_main": 0, "board_chinext": 0, "board_star": 0}
    for stock in targets:
        counts["board_" + board_type(stock)] = counts.get("board_" + board_type(stock), 0) + 1
    n = max(1, len(targets))
    counts["board_main_ratio"] = counts.get("board_main", 0) / float(n)
    counts["board_chinext_ratio"] = counts.get("board_chinext", 0) / float(n)
    counts["board_star_ratio"] = counts.get("board_star", 0) / float(n)
    counts["board_hhi"] = sum((counts[k] / float(n)) ** 2 for k in ["board_main", "board_chinext", "board_star"])
    return counts


def calc_portfolio_return(month_df, targets, col):
    if col is None or not targets:
        return np.nan
    vals = pd.to_numeric(month_df[month_df[STOCK_COL].astype(str).isin(targets)][col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return float(vals.mean()) if len(vals) else np.nan


def calc_turnover(prev_targets, targets):
    if not targets:
        return 0.0, 0
    if prev_targets is None:
        return 1.0, 0
    overlap = len(set(prev_targets) & set(targets))
    return 1.0 - overlap / float(max(1, len(targets))), overlap


def stable_text_seed(text):
    total = 0
    for i, ch in enumerate(str(text)):
        total += (i + 1) * ord(ch)
    return int(total % 100000)


def random_percentile_board_cap(month_df, actual_ret, ret_col, seed_key):
    if ret_col is None or pd.isnull(actual_ret):
        return np.nan
    d = month_df.dropna(subset=[ret_col]).copy().reset_index(drop=True)
    if d.empty:
        return np.nan
    rets = pd.to_numeric(d[ret_col], errors="coerce").values.astype(float)
    stocks = d[STOCK_COL].astype(str).values
    base_idx = np.arange(len(d), dtype=np.int32)
    rng = np.random.RandomState(seed_key)
    vals = []
    for _ in progress_iter(range(RANDOM_SIM_N), total=RANDOM_SIM_N, desc="random", leave=False):
        perm = rng.permutation(base_idx)
        picked_stocks = build_board_capped_targets([stocks[i] for i in perm], STOCK_NUM, BOARD_CAPS)
        picked_set = set(picked_stocks)
        picked_ret = pd.to_numeric(d[d[STOCK_COL].astype(str).isin(picked_set)][ret_col], errors="coerce").dropna()
        if len(picked_ret):
            vals.append(float(picked_ret.mean()))
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna()
    return float((vals <= actual_ret).mean()) if len(vals) else np.nan


In [ ]:
def train_fold_model(train_df, fold_row):
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, available_features)
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED + stable_text_seed(fold_row["fold_id"])
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix for " + str(fold_row["fold_id"]))
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=max(1, int(FIXED_ITER)))
    train_pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, train_pred)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, fill_values)
    valid_pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1) if len(X_valid) else []
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred) if len(X_valid) else np.nan
    return {
        "model": model,
        "feature_cols": feature_cols,
        "removed_cols": removed_cols,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
        "diag_rank_ic": diag_rank_ic,
    }


def score_fold(test_df, trained):
    X = test_df.reindex(columns=trained["feature_cols"]).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(trained["fill_values"]).fillna(0)
    out = test_df.copy()
    out["score"] = np.asarray(trained["model"].predict(X[trained["feature_cols"]], num_iteration=FIXED_ITER)).reshape(-1)
    out["score_rank_pct"] = out.groupby(DATE_COL)["score"].rank(pct=True)
    out["realized_rank_pct"] = out.groupby(DATE_COL)[TARGET_COL].rank(pct=True)
    return out


def evaluate_one_fold(fold_row, prev_targets=None):
    train_df = df_all[(df_all[DATE_COL] >= fold_row["train_start"]) & (df_all[DATE_COL] <= fold_row["train_end"])].copy()
    if not USE_LEGACY_UNSEALED_BOUNDARY and "next_date" in train_df.columns:
        train_df = train_df[train_df["next_date"] <= fold_row["train_end"]].copy()
    test_df = df_all[(df_all[DATE_COL] >= fold_row["test_start"]) & (df_all[DATE_COL] <= fold_row["test_end"])].copy()
    if train_df.empty or test_df.empty:
        return pd.DataFrame(), pd.DataFrame(), dict(prev_targets=prev_targets)

    trained = train_fold_model(train_df, fold_row)
    scored = score_fold(test_df, trained)
    scored["fold_set"] = fold_row["fold_set"]
    scored["fold_id"] = fold_row["fold_id"]
    scored["train_start"] = fold_row["train_start"]
    scored["train_end"] = fold_row["train_end"]
    scored["test_start"] = fold_row["test_start"]
    scored["test_end"] = fold_row["test_end"]

    rows = []
    for dt, month_df in progress_iter(list(scored.groupby(DATE_COL)), desc="months %s" % fold_row["fold_id"], leave=False):
        m = month_df.dropna(subset=[TARGET_COL, "score"]).copy()
        if m.empty:
            continue
        sorted_stocks = list(m.sort_values("score", ascending=False)[STOCK_COL].astype(str))
        targets = build_board_capped_targets(sorted_stocks, STOCK_NUM, BOARD_CAPS)
        gross_alpha_ret = calc_portfolio_return(m, targets, ALPHA_RET_COL)
        gross_raw_ret = calc_portfolio_return(m, targets, RAW_RET_COL) if RAW_RET_COL is not None else gross_alpha_ret
        benchmark_ret = get_month_benchmark_return(m, BENCH_RET_COL)
        turnover, overlap = calc_turnover(prev_targets, targets)
        actual_top20 = set(m.sort_values(TARGET_COL, ascending=False).head(20)[STOCK_COL].astype(str))
        actual_top10 = set(m.sort_values(TARGET_COL, ascending=False).head(10)[STOCK_COL].astype(str))
        selected_rows = m.set_index(STOCK_COL).reindex(targets)
        if not pd.isnull(gross_raw_ret) and not pd.isnull(benchmark_ret):
            excess_ret = gross_raw_ret - benchmark_ret
        else:
            excess_ret = gross_alpha_ret
        seed_key = int(pd.Timestamp(dt).strftime("%Y%m%d")) + stable_text_seed(fold_row["fold_id"])
        board = summarize_target_board(targets)
        row = {
            "fold_set": fold_row["fold_set"],
            "fold_id": fold_row["fold_id"],
            "train_start": fold_row["train_start"],
            "train_end": fold_row["train_end"],
            "test_start": fold_row["test_start"],
            "test_end": fold_row["test_end"],
            DATE_COL: dt,
            "portfolio_rule": PORTFOLIO_RULE,
            "stock_num": STOCK_NUM,
            "board_caps": BOARD_CAPS_TEXT,
            "target_count": len(targets),
            "gross_alpha_ret": gross_alpha_ret,
            "gross_raw_ret": gross_raw_ret,
            "benchmark_ret": benchmark_ret,
            "gross_excess_ret": excess_ret,
            "turnover": turnover,
            "target_overlap_prev": overlap,
            "rank_ic": safe_rank_ic(m["score"], m[TARGET_COL]),
            "target_avg_realized_rank": float(selected_rows["realized_rank_pct"].mean()),
            "selected_hit_real_top10": len(set(targets) & actual_top10),
            "selected_hit_real_top20": len(set(targets) & actual_top20),
            "selected_hit_rate_real_top20": len(set(targets) & actual_top20) / float(max(1, len(targets))),
            "random_alpha_percentile": random_percentile_board_cap(m, gross_alpha_ret, ALPHA_RET_COL, seed_key),
            "random_raw_percentile": random_percentile_board_cap(m, gross_raw_ret, RAW_RET_COL, seed_key + 17) if RAW_RET_COL is not None else np.nan,
            "targets": ",".join(targets),
        }
        row.update(board)
        rows.append(row)
        prev_targets = list(targets)

    monthly = pd.DataFrame(rows)
    meta = dict(fold_row)
    meta.update({
        "train_rows": trained["train_rows"],
        "feature_count": len(trained["feature_cols"]),
        "removed_feature_count": len(trained["removed_cols"]),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": trained["diag_rank_ic"],
        "feature_cols": ",".join(trained["feature_cols"]),
        "removed_features": ",".join(trained["removed_cols"]),
    })
    return monthly, pd.DataFrame([meta]), dict(prev_targets=prev_targets)


In [ ]:
monthly_parts = []
fold_meta_parts = []

for fold_set, gplan in progress_iter(list(fold_plan_df.groupby("fold_set")), desc="fold sets"):
    prev_targets = None
    gplan = gplan.sort_values("test_start").copy()
    for _, fold_row in progress_iter(list(gplan.iterrows()), total=len(gplan), desc="folds %s" % fold_set):
        monthly_one, meta_one, state = evaluate_one_fold(fold_row.to_dict(), prev_targets=prev_targets)
        prev_targets = state.get("prev_targets")
        if not monthly_one.empty:
            monthly_parts.append(monthly_one)
        if not meta_one.empty:
            fold_meta_parts.append(meta_one)

oos_monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
fold_meta_df = pd.concat(fold_meta_parts, ignore_index=True, sort=False) if fold_meta_parts else pd.DataFrame()

if oos_monthly_df.empty:
    raise ValueError("empty OOS monthly result")

oos_monthly_df = oos_monthly_df.sort_values(["fold_set", DATE_COL]).reset_index(drop=True)
oos_monthly_df["oos_nav"] = oos_monthly_df.groupby("fold_set")["gross_excess_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
oos_monthly_df["oos_drawdown"] = oos_monthly_df.groupby("fold_set")["gross_excess_ret"].transform(lambda s: (calc_nav(s) / calc_nav(s).cummax() - 1.0).values if len(s) else s)

display(fold_meta_df.head())
display(oos_monthly_df.tail(20))


In [ ]:
def summarize_fold_monthly(g):
    st = summarize_returns(g.sort_values(DATE_COL)["gross_excess_ret"])
    out = {
        "fold_set": g["fold_set"].iloc[0],
        "fold_id": g["fold_id"].iloc[0],
        "train_start": g["train_start"].iloc[0],
        "train_end": g["train_end"].iloc[0],
        "test_start": g["test_start"].iloc[0],
        "test_end": g["test_end"].iloc[0],
        "months": st["months"],
        "fold_cum_excess_ret": st["cum_ret"],
        "fold_mean_excess_ret": st["mean_ret"],
        "fold_win_rate": st["win_rate"],
        "fold_max_drawdown": st["max_drawdown"],
        "rank_ic_mean": float(pd.to_numeric(g["rank_ic"], errors="coerce").mean()),
        "random_alpha_percentile_mean": float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").mean()),
        "selected_hit_rate_real_top20_mean": float(pd.to_numeric(g["selected_hit_rate_real_top20"], errors="coerce").mean()),
        "avg_turnover": float(pd.to_numeric(g["turnover"], errors="coerce").mean()),
        "avg_board_hhi": float(pd.to_numeric(g["board_hhi"], errors="coerce").mean()),
        "avg_chinext_ratio": float(pd.to_numeric(g["board_chinext_ratio"], errors="coerce").mean()),
        "avg_star_ratio": float(pd.to_numeric(g["board_star_ratio"], errors="coerce").mean()),
    }
    return pd.Series(out)


fold_summary_df = oos_monthly_df.groupby(["fold_set", "fold_id"]).apply(summarize_fold_monthly).reset_index(drop=True)

def summarize_fold_set(g):
    ret_st = summarize_returns(g.sort_values(DATE_COL)["gross_excess_ret"])
    fold_g = fold_summary_df[fold_summary_df["fold_set"] == g["fold_set"].iloc[0]].copy()
    positive_fold_rate = float((fold_g["fold_cum_excess_ret"] > 0).mean()) if len(fold_g) else np.nan
    robust_pass = bool(
        ret_st["months"] >= PASS_MIN_MONTHS
        and ret_st["cum_ret"] > 0
        and ret_st["win_rate"] >= PASS_MIN_WIN_RATE
        and positive_fold_rate >= PASS_MIN_POSITIVE_FOLD_RATE
        and float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").mean()) >= PASS_MIN_RANDOM_PERCENTILE
        and ret_st["max_drawdown"] >= PASS_MAX_DRAWDOWN
    )
    return pd.Series({
        "fold_set": g["fold_set"].iloc[0],
        "portfolio_rule": PORTFOLIO_RULE,
        "stock_num": STOCK_NUM,
        "board_caps": BOARD_CAPS_TEXT,
        "folds": int(fold_g["fold_id"].nunique()),
        "months": ret_st["months"],
        "oos_cum_excess_ret": ret_st["cum_ret"],
        "oos_ann_excess_ret": ret_st["ann_ret"],
        "oos_mean_monthly_excess_ret": ret_st["mean_ret"],
        "oos_win_rate": ret_st["win_rate"],
        "oos_max_drawdown": ret_st["max_drawdown"],
        "oos_sharpe12": ret_st["sharpe12"],
        "positive_fold_rate": positive_fold_rate,
        "worst_fold_cum_excess_ret": float(fold_g["fold_cum_excess_ret"].min()) if len(fold_g) else np.nan,
        "median_fold_cum_excess_ret": float(fold_g["fold_cum_excess_ret"].median()) if len(fold_g) else np.nan,
        "rank_ic_mean": float(pd.to_numeric(g["rank_ic"], errors="coerce").mean()),
        "rank_ic_positive_rate": float((pd.to_numeric(g["rank_ic"], errors="coerce") > 0).mean()),
        "avg_random_alpha_percentile": float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").mean()),
        "p25_random_alpha_percentile": float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").quantile(0.25)),
        "avg_selected_hit_rate_real_top20": float(pd.to_numeric(g["selected_hit_rate_real_top20"], errors="coerce").mean()),
        "avg_turnover": float(pd.to_numeric(g["turnover"], errors="coerce").mean()),
        "avg_board_hhi": float(pd.to_numeric(g["board_hhi"], errors="coerce").mean()),
        "avg_chinext_ratio": float(pd.to_numeric(g["board_chinext_ratio"], errors="coerce").mean()),
        "avg_star_ratio": float(pd.to_numeric(g["board_star_ratio"], errors="coerce").mean()),
        "robust_pass": robust_pass,
    })

fold_set_summary_df = oos_monthly_df.groupby("fold_set").apply(summarize_fold_set).reset_index(drop=True)

yearly_rows = []
tmp = oos_monthly_df.copy()
tmp["year"] = pd.to_datetime(tmp[DATE_COL]).dt.year
for (fold_set, year), g in tmp.groupby(["fold_set", "year"]):
    st = summarize_returns(g.sort_values(DATE_COL)["gross_excess_ret"])
    yearly_rows.append({
        "fold_set": fold_set,
        "year": int(year),
        "months": st["months"],
        "cum_excess_ret": st["cum_ret"],
        "mean_excess_ret": st["mean_ret"],
        "win_rate": st["win_rate"],
        "max_drawdown": st["max_drawdown"],
        "rank_ic_mean": float(pd.to_numeric(g["rank_ic"], errors="coerce").mean()),
        "avg_random_alpha_percentile": float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").mean()),
    })
yearly_summary_df = pd.DataFrame(yearly_rows).sort_values(["fold_set", "year"])

display(fold_set_summary_df)
display(fold_summary_df)
display(yearly_summary_df)


In [ ]:
oos_monthly_df.to_csv(OUT_DIR / "v64_walk_forward_oos_monthly.csv", index=False)
fold_meta_df.to_csv(OUT_DIR / "v64_walk_forward_fold_meta.csv", index=False)
fold_plan_df.to_csv(OUT_DIR / "v64_walk_forward_fold_plan.csv", index=False)
fold_summary_df.to_csv(OUT_DIR / "v64_walk_forward_fold_summary.csv", index=False)
fold_set_summary_df.to_csv(OUT_DIR / "v64_walk_forward_summary.csv", index=False)
yearly_summary_df.to_csv(OUT_DIR / "v64_walk_forward_yearly_summary.csv", index=False)

print("saved outputs:")
for p in sorted(OUT_DIR.glob("v64_*.csv")):
    print("-", p)


## 读数规则

优先看 `v64_walk_forward_summary.csv`：

- `robust_pass=True`：不是证明可实盘，只说明当前算法通过基础 OOS 鲁棒性门槛。
- `positive_fold_rate`：不同训练窗口是否多数为正。
- `worst_fold_cum_excess_ret`：最差窗口是否可以接受。
- `avg_random_alpha_percentile` / `p25_random_alpha_percentile`：是否稳定优于同板块约束随机组合。
- `rank_ic_mean` 和 `rank_ic_positive_rate`：模型排序健康度。
- `avg_board_hhi`：组合构建是否仍过度压在某个板块。

失败处理：

- 如果 anchored 通过但 rolling60 不通过：说明长期历史对模型有帮助，近年局部训练不稳。
- 如果 rolling60 通过但 anchored 不通过：说明旧样本可能拖累，需要考虑训练窗口老化。
- 如果两者都不通过：先不要加模型复杂度，回到特征/标签/组合构建做审计。
- 如果两者都通过：下一步把 V61 的 JQ-like 对齐结果作为主验收，再考虑真实 JoinQuant 回测。
